In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.gridspec as gridspec

import numpy as np

In [ ]:
df_order= pd.read_csv('../dataset/orders.csv')
df_payments= pd.read_csv('../dataset/payments.csv', low_memory=False)

df = df_order.merge(df_payments, on="order_id", how="left")

In [ ]:
df.columns = df.columns.str.strip()

In [ ]:
df["order_status"].value_counts()

In [ ]:
df["is_cancelled"] = (df["order_status"] == "cancelled").astype(int)
df["is_installment"] = df["installments"] > 1

In [ ]:
df.head()

In [ ]:
df = df.drop(columns=["payment_method_x"])

In [ ]:
df = df.rename(columns={"payment_method_y": "payments_method"})

In [ ]:
df.head()


In [ ]:
df["payments_method"].value_counts(normalize=True)

In [ ]:
payment_dist = {
    "CreditCard": 0.550823,
    "PayPal": 0.149963,
    "COD": 0.149442,
    "Apple Pay": 0.100106,
    "Bank Transfer": 0.049666
}

# convert sang dataframe giống cat_data của bạn
pay_data = pd.DataFrame(list(payment_dist.items()), columns=["payment_method", "share"])

# sort
pay_data = pay_data.sort_values("share", ascending=False)

plt.figure(figsize=(10,6))

sns.barplot(
    data=pay_data,
    x="share",
    y="payment_method",
    hue="payment_method",
    palette="viridis",
    legend=False
)

plt.title("Payment Method Distribution (%)")
plt.xlabel("Share")
plt.ylabel("Payment Method")

plt.show()

### Credit Card dominates with ~55% share, indicating strong user preference but also creating a high dependency risk; diversification of payment channels is required to improve system resilience and long-term growth stability.

In [ ]:
cancel_rate = df.groupby("payments_method")["is_cancelled"].mean().reset_index()
cancel_rate = cancel_rate.sort_values("is_cancelled", ascending=False)

In [ ]:

sns.set_style("whitegrid")

plt.figure(figsize=(10,6), facecolor="white")
ax = plt.gca()
ax.set_facecolor("white")

sns.barplot(
    data=cancel_rate,
    x="is_cancelled",
    y="payments_method",
    hue="payments_method",
    palette="viridis",
    legend=False
)

plt.title("Cancel Rate by Payment Method")
plt.xlabel("Cancel Rate")
plt.ylabel("Payment Method")

plt.show()

### Payment method có ảnh hưởng rõ rệt đến tỷ lệ huỷ đơn. COD thường có tỷ lệ huỷ cao nhất do người dùng không cam kết thanh toán trước, trong khi Credit Card và Apple Pay có tỷ lệ huỷ thấp hơn nhờ cơ chế thanh toán ngay lập tức. Điều này cho thấy các phương thức thanh toán số (digital payments) giúp tăng độ chắc chắn của đơn hàng, giảm rủi ro huỷ và cải thiện chất lượng doanh thu

In [ ]:
install_analysis = df.groupby("is_installment")["payment_value"].mean().reset_index()

In [ ]:
install_analysis.head()

In [ ]:
plt.figure(figsize=(6,5), facecolor="white")
ax = plt.gca()
ax.set_facecolor("white")

sns.barplot(
    data=install_analysis,
    x="is_installment",
    y="payment_value",
    hue="is_installment",
    palette="viridis",
    legend=False
)

plt.title("Installments vs Order Value")
plt.xlabel("Installment (False=0, True=1)")
plt.ylabel("Average Order Value")

plt.show()

### Installment orders exhibit slightly higher average order value compared to non-installment orders (~24.3K vs ~24.1K), indicating a mild but consistent uplift effect on customer spending behavior, particularly in higher-value segments.

In [ ]:
rev_by_pay = df.groupby("payments_method")["payment_value"].agg(["sum", "mean", "count"]).reset_index()
rev_by_pay = rev_by_pay.sort_values("sum", ascending=False)

In [ ]:


sns.set_style("whitegrid")

plt.figure(figsize=(10,6), facecolor="white")
ax = plt.gca()
ax.set_facecolor("white")

sns.barplot(
    data=rev_by_pay,
    x="sum",
    y="payments_method",
    hue="payments_method",
    palette="viridis",
    legend=False
)

plt.title("Revenue by Payment Method")
plt.xlabel("Total Revenue")
plt.ylabel("Payment Method")

plt.show()

### Credit Card is the dominant revenue contributor, accounting for the majority of total transaction value, while other payment methods collectively form a fragmented but strategically important secondary ecosystem.

In [ ]:
risk = df.groupby("payments_method").agg({
    "is_cancelled": "mean",
    "payment_value": "mean",
    "installments": "mean"
}).reset_index()

risk = risk.sort_values("is_cancelled", ascending=False)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

data = [
    {"label": "COD",           "x": 24275, "y": 0.160, "r": 600, "color": "#A32D2D", "installments": 3.88},
    {"label": "PayPal",        "x": 24365, "y": 0.081, "r": 500, "color": "#185FA5", "installments": 3.88},
    {"label": "Apple Pay",     "x": 24152, "y": 0.080, "r": 350, "color": "#1D9E75", "installments": 3.85},
    {"label": "Credit Card",   "x": 24158, "y": 0.081, "r": 400, "color": "#639922", "installments": 3.87},
    {"label": "Bank Transfer", "x": 24218, "y": 0.080, "r": 350, "color": "#BA7517", "installments": 3.85},
]

fig, ax = plt.subplots(figsize=(11, 6.5), facecolor="white")
ax.set_facecolor("#FAFAF8")

# vùng an toàn
ax.axhspan(0, 0.10, color="#1D9E75", alpha=0.05, zorder=0)
ax.axhline(0.12, color="#F09595", lw=1.2, ls="--", alpha=0.8, zorder=1)
ax.text(24420, 0.121, "ngưỡng rủi ro cao", fontsize=9,
        color="#A32D2D", va="bottom", ha="right", style="italic")

# vẽ bong bóng
for d in data:
    ax.scatter(d["x"], d["y"],
               s=d["r"],
               color=d["color"],
               alpha=0.25,
               edgecolors=d["color"],
               linewidths=2,
               zorder=3)
    # label tên
    offset_y = 0.006 if d["label"] in ("Apple Pay", "Bank Transfer") else -0.007
    ax.text(d["x"], d["y"] + offset_y, d["label"],
            fontsize=10, fontweight="bold",
            color=d["color"], ha="center", va="center", zorder=4)
    # label cancel rate
    ax.text(d["x"], d["y"] + offset_y - 0.0065,
            f"{d['y']*100:.1f}%",
            fontsize=8.5, color="#888780", ha="center", va="top", zorder=4)

# lưới tinh tế
ax.grid(True, color="#D3D1C7", linewidth=0.5, linestyle="--", alpha=0.6)
ax.set_axisbelow(True)

# trục
ax.set_xlabel("Average Order Value (AOV)", fontsize=11, color="#5F5E5A", labelpad=10)
ax.set_ylabel("Cancel Rate", fontsize=11, color="#5F5E5A", labelpad=10)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v*100:.0f}%"))
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{int(v):,}"))
ax.tick_params(colors="#888780", labelsize=9)
for spine in ax.spines.values():
    spine.set_edgecolor("#D3D1C7")
    spine.set_linewidth(0.8)

ax.set_xlim(24110, 24430)
ax.set_ylim(0.04, 0.20)

# legend kích thước bong bóng
for size, label in [(350, "ít trả góp"), (500, "trung bình"), (600, "nhiều trả góp")]:
    ax.scatter([], [], s=size, color="#B4B2A9", alpha=0.4,
               edgecolors="#888780", linewidths=1.5, label=label)
ax.legend(title="Số installments", title_fontsize=9,
          fontsize=8.5, framealpha=0.6, loc="upper left",
          edgecolor="#D3D1C7", labelcolor="#5F5E5A")

ax.set_title("Payment Method Risk Profile",
             fontsize=14, fontweight="bold", color="#2C2C2A",
             pad=16, loc="left")
ax.text(0.01, 1.01,
        "Cancel rate vs AOV — kích thước bong bóng = số lần trả góp",
        transform=ax.transAxes, fontsize=9, color="#888780")

plt.tight_layout()
plt.savefig("payment_risk_profile.png", dpi=150, bbox_inches="tight")
plt.show()

### COD is the only real problem here. At 16% cancel rate — double every other method — it's a textbook moral hazard story: no upfront commitment, no friction to walk away. What makes it costly is that COD's AOV isn't materially lower than the rest of the cohort, so you're absorbing return logistics on mid-ticket orders, not small impulse buys. Every other payment method clusters tightly around 8% with negligible AOV spread, which rules out order value as a driver and points squarely at pre-commitment as the differentiating variable. Fix COD, and you've essentially fixed the cancellation problem

In [ ]:
df_payments["installments"] = df_payments["installments"].astype(int)

def installment_group(n):
    if n == 1:    return "1 kỳ (trả thẳng)"
    elif n <= 3:  return "2–3 kỳ"
    elif n <= 6:  return "4–6 kỳ"
    elif n <= 12: return "7–12 kỳ"
    else:         return "12+ kỳ"

df_payments["inst_group"] = df_payments["installments"].apply(installment_group)
GROUP_ORDER = ["1 kỳ (trả thẳng)", "2–3 kỳ", "4–6 kỳ", "7–12 kỳ"]

# Kiểm tra
print(df_payments[["installments", "inst_group"]].head(10))
print(df_payments["inst_group"].value_counts())

In [ ]:
# Plot 1
fig, ax = plt.subplots(figsize=(8, 5), facecolor="white")
ax.set_facecolor("#FAFAF8")

aov_group = (df_payments.groupby("inst_group")["payment_value"]
             .mean().reindex(GROUP_ORDER))
colors = [C_GREEN if v == aov_group.max() else
          C_RED   if v == aov_group.min() else C_BLUE
          for v in aov_group]
bars = ax.bar(aov_group.index, aov_group.values,
              color=colors, alpha=0.8, edgecolor="white")
for bar, val in zip(bars, aov_group.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f"{val:,.0f}", ha="center", fontsize=9, color="#2C2C2A")
ax.set_title("Avg AOV theo Nhóm Installments", fontweight="bold", fontsize=12)
ax.set_ylabel("Avg Payment Value")
plt.setp(ax.get_xticklabels(), rotation=15, ha="right")
plt.tight_layout()
plt.show()

### installments không phải lever kích cầu hiệu quả trong dataset này — khách hàng chọn trả góp không phải vì muốn mua hàng đắt hơn, mà nhiều khả năng vì lý do dòng tiền cá nhân. Nếu muốn dùng installments để tăng AOV, cần kết hợp thêm incentive rõ ràng hơn như miễn lãi 0% cho đơn trên ngưỡng giá trị nhất định.

In [ ]:
shipments = pd.read_csv("../dataset/shipments.csv")
inventory = pd.read_csv("../dataset/inventory.csv")

In [ ]:
shipments["ship_date"] = pd.to_datetime(shipments["ship_date"])
shipments["delivery_date"] = pd.to_datetime(shipments["delivery_date"])

shipments["delivery_days"] = (
    shipments["delivery_date"] - shipments["ship_date"]n
).dt.days

In [ ]:
inventory["stockout_days"].describe()
print(df.columns)

In [ ]:
plt.figure()
plt.hist(inventory["stockout_days"], bins=30)
plt.title("Stockout Days Distribution")
plt.xlabel("Days")
plt.ylabel("Frequency")
plt.show()
plt.figure(figsize=(8,2))

plt.boxplot(inventory["stockout_days"], vert=False, patch_artist=True)

plt.title("Stockout Days - Outlier View")
plt.xlabel("Days")

plt.show()

### Phân phối stockout lệch phải mạnh — phần lớn sự kiện hết hàng được giải quyết trong vòng 0–3 ngày, cho thấy gián đoạn cung ứng chủ yếu đến từ độ trễ bổ sung hàng định kỳ, không phải lỗi hệ thống tồn kho. Tuy nhiên, outlier view mới là phần đáng lo: đuôi phân phối kéo dài đến ~28 ngày, tức một tập nhỏ SKU đang hết hàng lâu hơn mức trung vị cả chục lần — đây là nơi thất thoát doanh thu và rủi ro mất khách hàng thực sự tích lũy. Median rơi vào 1–2 ngày trông có vẻ lành mạnh, nhưng nếu báo cáo tồn kho dùng mean thay vì median thì con số tổng hợp sẽ bị kéo lên đáng kể bởi cái đuôi dài đó — đúng kiểu vấn đề bị che khuất trong aggregate report và chỉ lộ ra khi bóc tách theo danh mục sản phẩm hoặc lead time nhà cung cấp.

In [ ]:
inventory["avg_daily_demand"] = (
    inventory["units_sold"] / 30   # nếu snapshot monthly
)
inventory["lost_demand"] = (
    inventory["stockout_days"] * inventory["avg_daily_demand"]
)
avg_price_proxy = inventory["units_sold"].sum() / inventory["stock_on_hand"].sum()

inventory["lost_revenue"] = (
    inventory["lost_demand"] * avg_price_proxy
)

In [ ]:
top_loss = inventory.sort_values("lost_revenue", ascending=False).head(10)


plt.figure(figsize=(10,5))
plt.bar(top_loss["product_name"], top_loss["lost_revenue"])
plt.xticks(rotation=45, ha="right")
plt.title("Top 10 SKUs by Lost Revenue (Stockout Impact)")
plt.show()

# Đánh giá hiệu quả lưu kho thông qua tỷ lệ hàng bán (sell_through_rate) và cảnh báo tồn kho vượt mức (overstock_flag) [cite: 121].


In [ ]:
# ── Sell-through Rate ──────────────────────────────────────────
# = units_sold / (units_sold + stock_on_hand)
# Ý nghĩa: tỷ lệ hàng nhập về đã được bán ra
inventory["sell_through_rate"] = inventory["units_sold"] / (
    inventory["units_sold"] + inventory["stock_on_hand"]
).replace(0, np.nan)

inventory["sell_through_rate"] = inventory["sell_through_rate"].clip(0, 1)

# ── Overstock Flag ─────────────────────────────────────────────
# Flag = 1 nếu stock_on_hand > 2x avg_daily_demand * 30 ngày
# (tức là tồn kho đủ dùng hơn 60 ngày — điều chỉnh ngưỡng tuỳ business)
OVERSTOCK_THRESHOLD_DAYS = 60

inventory["overstock_flag"] = (
    inventory["stock_on_hand"] > inventory["avg_daily_demand"] * OVERSTOCK_THRESHOLD_DAYS
).astype(int)

# ── Summary ────────────────────────────────────────────────────
summary = inventory.groupby("category").agg(
    avg_sell_through  = ("sell_through_rate", "mean"),
    overstock_pct     = ("overstock_flag", "mean"),
    overstock_skus    = ("overstock_flag", "sum"),
    total_skus        = ("product_id", "nunique"),
).reset_index()

summary["avg_sell_through"] = summary["avg_sell_through"].round(3)
summary["overstock_pct"]    = (summary["overstock_pct"] * 100).round(1)

print(summary.sort_values("overstock_pct", ascending=False))


In [ ]:


# ── Summary ────────────────────────────────────────────────────
summary = inventory.groupby("category").agg(
    avg_sell_through = ("sell_through_rate", "mean"),
    overstock_pct    = ("overstock_flag", "mean"),
    overstock_skus   = ("overstock_flag", "sum"),
    total_skus       = ("product_id", "nunique"),
).reset_index()

summary["avg_sell_through"] = summary["avg_sell_through"].round(3)
summary["overstock_pct"]    = (summary["overstock_pct"] * 100).round(1)

# ── Setup ──────────────────────────────────────────────────────
C_BLUE, C_RED, C_GREEN = "#185FA5", "#A32D2D", "#1D9E75"
C_GRAY, C_BG, C_GRID   = "#888780", "#FAFAF8", "#E0DED8"

def style_ax(ax):
    ax.set_facecolor(C_BG)
    ax.grid(True, color=C_GRID, linewidth=0.5, linestyle="--", alpha=0.7)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_edgecolor(C_GRID); spine.set_linewidth(0.8)
    ax.tick_params(colors=C_GRAY, labelsize=9)
    ax.xaxis.label.set_color(C_GRAY)
    ax.yaxis.label.set_color(C_GRAY)
    ax.title.set_color("#2C2C2A")

fig = plt.figure(figsize=(14, 10), facecolor="white")
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)

# ── Plot 1: Sell-through Distribution ─────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(inventory["sell_through_rate"].dropna(), bins=30,
         color=C_BLUE, alpha=0.75, edgecolor="white", linewidth=0.5)
ax1.axvline(inventory["sell_through_rate"].median(), color=C_RED,
            lw=1.5, ls="--",
            label=f'Median: {inventory["sell_through_rate"].median():.2f}')
ax1.set_title("Sell-through Rate Distribution", fontweight="bold", fontsize=11)
ax1.set_xlabel("Sell-through Rate")
ax1.set_ylabel("Frequency")
ax1.legend(fontsize=9, framealpha=0.5)
style_ax(ax1)

# ── Plot 2: Avg Sell-through by Category ──────────────────────
ax2 = fig.add_subplot(gs[0, 1])
cat_str = summary.sort_values("avg_sell_through")
colors  = [C_GREEN if v >= 0.6 else C_RED for v in cat_str["avg_sell_through"]]
bars = ax2.barh(cat_str["category"], cat_str["avg_sell_through"],
                color=colors, alpha=0.8, edgecolor="white")
ax2.axvline(0.6, color=C_GRAY, lw=1, ls="--", alpha=0.6)
ax2.text(0.61, -0.6, "ngưỡng 60%", fontsize=8, color=C_GRAY, style="italic")
for bar, val in zip(bars, cat_str["avg_sell_through"]):
    ax2.text(val + 0.005, bar.get_y() + bar.get_height()/2,
             f"{val:.1%}", va="center", fontsize=8.5, color="#2C2C2A")
ax2.set_title("Avg Sell-through by Category", fontweight="bold", fontsize=11)
ax2.set_xlabel("Sell-through Rate")
ax2.set_xlim(0, 1.05)
style_ax(ax2)

# ── Plot 3: Overstock % by Category ───────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
cat_ov  = summary.sort_values("overstock_pct", ascending=False)
colors2 = [C_RED if v > 20 else C_BLUE for v in cat_ov["overstock_pct"]]
bars2   = ax3.bar(cat_ov["category"], cat_ov["overstock_pct"],
                  color=colors2, alpha=0.8, edgecolor="white")
for bar, val in zip(bars2, cat_ov["overstock_pct"]):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f"{val:.1f}%", ha="center", fontsize=8.5, color="#2C2C2A")
ax3.axhline(20, color=C_GRAY, lw=1, ls="--", alpha=0.6)
ax3.text(0, 20.5, "ngưỡng cảnh báo 20%", fontsize=8, color=C_GRAY, style="italic")
ax3.set_title("Overstock Rate by Category", fontweight="bold", fontsize=11)
ax3.set_ylabel("% SKUs bị overstock")
plt.setp(ax3.get_xticklabels(), rotation=20, ha="right")
style_ax(ax3)

# ── Plot 4: Scatter stock vs sell-through ─────────────────────
ax4 = fig.add_subplot(gs[1, 1])
sample = inventory.sample(min(3000, len(inventory)), random_state=42)
sc = ax4.scatter(
    sample["stock_on_hand"], sample["sell_through_rate"],
    c=sample["overstock_flag"], cmap="RdYlGn_r",
    alpha=0.35, s=15, edgecolors="none"
)
ax4.axhline(0.6, color=C_GRAY, lw=1, ls="--", alpha=0.6)
ax4.set_title("Stock on Hand vs Sell-through Rate", fontweight="bold", fontsize=11)
ax4.set_xlabel("Stock on Hand")
ax4.set_ylabel("Sell-through Rate")
ax4.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
cbar = plt.colorbar(sc, ax=ax4, pad=0.02)
cbar.set_label("Overstock Flag", fontsize=9, color=C_GRAY)
cbar.ax.tick_params(labelsize=8)
style_ax(ax4)

fig.suptitle("Inventory Efficiency: Sell-through Rate & Overstock Analysis",
             fontsize=14, fontweight="bold", color="#2C2C2A", y=1.01)

plt.savefig("inventory_efficiency.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show()

### Bức tranh tổng thể rất rõ ràng — đây là một hệ thống tồn kho đang hoạt động kém hiệu quả nghiêm trọng trên toàn danh mục. Sell-through rate median chỉ 0.11 (11%), tức trung bình chỉ 1 trong 9 đơn vị hàng nhập về được bán ra — phần còn lại nằm chết trong kho. Biểu đồ category confirm điều này: cả 4 nhóm Casual, GenZ, Streetwear, Outdoor đều dưới 17%, không có category nào chạm ngưỡng 60% — nghĩa là vấn đề không nằm ở một dòng hàng cụ thể mà là structural, toàn bộ portfolio đang được mua vào nhiều hơn thị trường hấp thụ được. Overstock rate càng xác nhận điều này: 85–91% SKU trên tất cả category đang bị gắn cờ overstock — con số này không còn là cảnh báo nữa, đây là trạng thái bình thường của hệ thống, điều đó có nghĩa là ngưỡng reorder point hoặc purchase order đang được set quá cao so với actual demand. Scatter plot chốt lại logic: phần lớn SKU tập trung ở góc trái dưới — tồn kho thấp và sell-through thấp — cho thấy vấn đề không phải thiếu hàng mà là demand yếu hoặc dự báo nhu cầu sai, dẫn đến toàn bộ hệ thống đang chôn vốn lưu động vào hàng không bán được.

# Phân tích thời gian giao hàng thực tế (delivery_date - ship_date) [cite: 75].


In [ ]:
shipments["ship_date"]      = pd.to_datetime(shipments["ship_date"])
shipments["delivery_date"]  = pd.to_datetime(shipments["delivery_date"])
shipments["delivery_days"]  = (
    shipments["delivery_date"] - shipments["ship_date"]
).dt.days

# Loại outlier âm (data error)
shipments = shipments[shipments["delivery_days"] >= 0]

# ── Summary ────────────────────────────────────────────────────
print(shipments["delivery_days"].describe().round(2))
print(f"\n% giao trong 3 ngày : {(shipments['delivery_days'] <= 3).mean():.1%}")
print(f"% giao trong 7 ngày : {(shipments['delivery_days'] <= 7).mean():.1%}")
print(f"% giao trên 14 ngày : {(shipments['delivery_days'] > 14).mean():.1%}")

In [ ]:
# Kiểm tra phân phối
print(shipments["delivery_days"].value_counts().sort_index())
print(shipments["delivery_days"].describe())

# Xem thử vài dòng
print(shipments[["ship_date", "delivery_date", "delivery_days"]].head(20))

### Delivery days phân phối đều từ 2–7 (uniform distribution)
→ Data được synthetic generated, không đủ cơ sở để rút insight logistics thực tế.

# Feature Engineering

In [ ]:
# Merge và drop cột trùng
df = df_order.merge(df_payments, on=["order_id", "payment_method"], how="left")

# Kiểm tra
print(df.columns.tolist())
print(df.shape)

In [ ]:
# ── Installment features ───────────────────────────────────────
df["is_installment"]     = (df["installments"] > 1).astype(int)
df["high_installment"]   = (df["installments"] >= 7).astype(int)
df["payment_per_inst"]   = df["payment_value"] / df["installments"].replace(0, np.nan)

# ── Payment method features ────────────────────────────────────
df["is_cod"]             = (df["payment_method"] == "cod").astype(int)
df["is_digital_payment"] = df["payment_method"].isin(
    ["credit_card", "paypal", "apple_pay"]).astype(int)
df["is_bank_transfer"]   = (df["payment_method"] == "bank_transfer").astype(int)

risk_map = {
    "cod":           0.16,
    "paypal":        0.081,
    "credit_card":   0.081,
    "apple_pay":     0.080,
    "bank_transfer": 0.080,
}
df["payment_risk_score"] = df["payment_method"].map(risk_map)

# ── Cancel features ────────────────────────────────────────────
df["is_cancelled"]   = (df["order_status"] == "cancelled").astype(int)
df["cancel_risk"]    = df["is_cod"] * df["payment_risk_score"]

# ── Inventory features ─────────────────────────────────────────
inv = inventory.copy()

inv["is_stockout"]           = (inv["stockout_days"] > 0).astype(int)
inv["stockout_revenue_risk"] = inv["stockout_days"] * inv["avg_daily_demand"]
inv["fill_rate_gap"]         = 1 - inv["fill_rate"]
inv["low_fill_rate"]         = (inv["fill_rate"] < inv["fill_rate"].quantile(0.25)).astype(int)
inv["sell_through_gap"]      = 1 - inv["sell_through_rate"]
inv["stock_health"]          = (
    inv["fill_rate"] * 0.4 +
    inv["sell_through_rate"] * 0.4 +
    (1 - inv["overstock_flag"]) * 0.2
)

SALE_MONTHS = {1, 4, 6, 11, 12}
inv["is_sale_month"] = inv["month"].isin(SALE_MONTHS).astype(int)
inv["fill_x_sale"]   = inv["fill_rate"] * inv["is_sale_month"]

inv = inv.sort_values(["product_id", "snapshot_date"])
inv["fill_rate_lag1"]     = inv.groupby("product_id")["fill_rate"].shift(1)
inv["stockout_days_lag1"] = inv.groupby("product_id")["stockout_days"].shift(1)
inv["sell_through_lag1"]  = inv.groupby("product_id")["sell_through_rate"].shift(1)
inv["fill_rate_roll3"]    = (inv.groupby("product_id")["fill_rate"]
                             .transform(lambda x: x.shift(1).rolling(3).mean()))
inv["stockout_roll3"]     = (inv.groupby("product_id")["stockout_days"]
                             .transform(lambda x: x.shift(1).rolling(3).mean()))

# ── Summary ────────────────────────────────────────────────────
print("── Payment features ──")
print(df[["is_cod", "is_installment", "payment_risk_score",
          "payment_per_inst", "cancel_risk"]].describe().round(3))

print("\n── Inventory features ──")
print(inv[["is_stockout", "fill_rate_gap", "stock_health",
           "stockout_revenue_risk", "fill_rate_lag1"]].describe().round(3))

In [ ]:
inv = inv.drop(columns=["fill_rate_gap", "fill_rate_lag1"])
from sklearn.preprocessing import MinMaxScaler
inv["stock_health_scaled"] = MinMaxScaler().fit_transform(
    inv[["stock_health"]])

In [117]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
from sklearn.preprocessing import MinMaxScaler


# ── Tạo join key theo tháng ────────────────────────────────────
df["order_date"] = pd.to_datetime(df["order_date"])
df["year"]       = df["order_date"].dt.year
df["month"]      = df["order_date"].dt.month

# Aggregate orders theo tháng — target là tổng revenue
monthly_orders = df.groupby(["year", "month"]).agg(
    Revenue              = ("payment_value", "sum"),
    avg_payment_value    = ("payment_value", "mean"),
    total_orders         = ("order_id", "count"),
    cancel_rate          = ("is_cancelled", "mean"),
    cod_rate             = ("is_cod", "mean"),
    installment_rate     = ("is_installment", "mean"),
    avg_payment_per_inst = ("payment_per_inst", "mean"),
    avg_risk_score       = ("payment_risk_score", "mean"),
    digital_rate         = ("is_digital_payment", "mean"),
).reset_index()

# Aggregate inventory theo tháng
monthly_inv = inv.groupby(["year", "month"]).agg(
    avg_fill_rate         = ("fill_rate", "mean"),
    avg_stockout_days     = ("stockout_days", "mean"),
    avg_sell_through      = ("sell_through_rate", "mean"),
    avg_stock_health      = ("stock_health_scaled", "mean"),
    total_lost_revenue    = ("lost_revenue", "sum"),
    avg_stockout_risk     = ("stockout_revenue_risk", "mean"),
    overstock_rate        = ("overstock_flag", "mean"),
    stockout_rate         = ("is_stockout", "mean"),
    sale_month            = ("is_sale_month", "max"),
    avg_fill_x_sale       = ("fill_x_sale", "mean"),
    avg_stockout_roll3    = ("stockout_roll3", "mean"),
    avg_fill_roll3        = ("fill_rate_roll3", "mean"),
).reset_index()

# ── Join ───────────────────────────────────────────────────────
data = monthly_orders.merge(monthly_inv, on=["year", "month"], how="left")

print(f"Shape sau join: {data.shape}")
print(data[["year", "month", "Revenue"]].head(10))

# ── Time features ──────────────────────────────────────────────
data["quarter"]      = ((data["month"] - 1) // 3) + 1
data["is_tet"]       = data["month"].isin([1, 2]).astype(int)
data["is_blackfri"]  = (data["month"] == 11).astype(int)
data["is_xmas"]      = (data["month"] == 12).astype(int)

# Lag features
data = data.sort_values(["year", "month"]).reset_index(drop=True)
for lag in [1, 2, 3]:
    data[f"revenue_lag{lag}"]      = data["Revenue"].shift(lag)
    data[f"orders_lag{lag}"]       = data["total_orders"].shift(lag)
    data[f"cancel_rate_lag{lag}"]  = data["cancel_rate"].shift(lag)

# Rolling
data["revenue_roll3"] = data["Revenue"].shift(1).rolling(3).mean()
data["revenue_roll6"] = data["Revenue"].shift(1).rolling(6).mean()
data["orders_roll3"]  = data["total_orders"].shift(1).rolling(3).mean()

# ── Features & Target ──────────────────────────────────────────
FEATURES = [
    # Time
    "year", "month", "quarter",
    "is_tet", "is_blackfri", "is_xmas", "sale_month",
    # Payment
    "avg_payment_value", "total_orders", "cancel_rate",
    "cod_rate", "installment_rate", "digital_rate",
    "avg_payment_per_inst", "avg_risk_score",
    # Inventory
    "avg_fill_rate", "avg_stockout_days", "avg_sell_through",
    "avg_stock_health", "total_lost_revenue", "avg_stockout_risk",
    "overstock_rate", "stockout_rate", "avg_fill_x_sale",
    "avg_stockout_roll3", "avg_fill_roll3",
    # Lags
    "revenue_lag1", "revenue_lag2", "revenue_lag3",
    "orders_lag1", "cancel_rate_lag1",
    "revenue_roll3", "revenue_roll6", "orders_roll3",
]

TARGET = "Revenue"

data_clean = data.dropna(subset=FEATURES)
print(f"\nShape sau dropna: {data_clean.shape}")

X = data_clean[FEATURES]
y = data_clean[TARGET]

# ── Train/Test split theo thời gian ───────────────────────────
split = int(len(data_clean) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

# ── Model ──────────────────────────────────────────────────────
model = GradientBoostingRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    random_state=42
)
model.fit(X_train[FEATURES], y_train)

y_pred = model.predict(X_test[FEATURES])

print(f"MAE  : {mean_absolute_error(y_test, y_pred):,.0f}")
print(f"MAPE : {mean_absolute_percentage_error(y_test, y_pred):.2%}")


Shape sau join: (126, 23)
   year  month       Revenue
0  2012      7  1.304068e+08
1  2012      8  1.590892e+08
2  2012      9  1.293071e+08
3  2012     10  1.101857e+08
4  2012     11  9.818630e+07
5  2012     12  1.143226e+08
6  2013      1  9.128545e+07
7  2013      2  1.063607e+08
8  2013      3  1.415457e+08
9  2013      4  1.896518e+08

Shape sau dropna: (120, 39)
MAE  : 10,706,323
MAPE : 17.43%


In [116]:
df_daily = df_order.merge(df_payments, on=["order_id", "payment_method"], how="left")
df_daily["order_date"] = pd.to_datetime(df_daily["order_date"])

df_daily = df_daily.groupby("order_date").agg(
    Revenue = ("payment_value", "sum"),
    COGS    = ("payment_value", lambda x: x.sum() * 0.75),
).reset_index().rename(columns={"order_date": "Date"})

df_daily["Profit"] = df_daily["Revenue"] - df_daily["COGS"]
df_daily["month"]  = df_daily["Date"].dt.month


print(df_daily.shape)
print(df_daily["Date"].min(), "→", df_daily["Date"].max())
print(df_daily.head())

(3833, 5)
2012-07-04 00:00:00 → 2022-12-31 00:00:00
        Date     Revenue          COGS        Profit  month
0 2012-07-04  5123547.94  3.842661e+06  1.280887e+06      7
1 2012-07-05  2751773.45  2.063830e+06  6.879434e+05      7
2 2012-07-06  3054029.42  2.290522e+06  7.635074e+05      7
3 2012-07-07  2667930.94  2.000948e+06  6.669827e+05      7
4 2012-07-08  2360851.90  1.770639e+06  5.902130e+05      7


In [115]:
from tqdm import tqdm

# ── Helper function ────────────────────────────────────────────
def get_lag_safe(date, lag, col, hist_dict):
    lag_date = date - pd.Timedelta(days=lag)
    if lag_date in hist_dict:
        return hist_dict[lag_date][col]
    return 0

# ── Prep historical dict ───────────────────────────────────────
historical_df = df_daily[['Date', 'Revenue', 'Profit']].iloc[-370:].copy()
historical_df = historical_df.rename(columns={'Date': 'ds', 'Revenue': 'y', 'Profit': 'profit'})
historical_dict = historical_df.set_index('ds').to_dict('index')

# ── Static features từ training data ──────────────────────────
df_daily["is_weekend"]  = df_daily["Date"].dt.dayofweek.isin([5, 6]).astype(int)
df_daily["is_promoted"] = 0  # không có data promotion
df_daily["avg_discount_pct"] = 0
df_daily["max_discount_pct"] = 0

train_monthly_avg = df_daily.groupby("month")[
    ["is_promoted", "avg_discount_pct", "max_discount_pct"]
].mean().to_dict("index")

# ── Feature list ───────────────────────────────────────────────
features = [
    "month", "day_of_week", "is_weekend", "is_special", "is_payday",
    "rev_lag_1", "profit_lag_7", "profit_lag_365", "margin_rate_lag_7",
    "is_promoted", "avg_discount_pct", "max_discount_pct"
]

# ── Train model Revenue + COGS ────────────────────────────────
df_train = df_daily.copy()
df_train["day_of_week"]      = df_train["Date"].dt.dayofweek
df_train["is_special"]       = df_train["Date"].dt.strftime("%m-%d").isin(
    ["01-01","02-14","04-30","05-01","12-31","12-24","12-25"]).astype(int)
df_train["is_payday"]        = ((df_train["Date"].dt.day == 15) |
                                 df_train["Date"].dt.is_month_end).astype(int)
df_train["rev_lag_1"]        = df_train["Revenue"].shift(1)
df_train["profit_lag_7"]     = df_train["Profit"].shift(7)
df_train["profit_lag_365"]   = df_train["Profit"].shift(365)
df_train["margin_rate_lag_7"] = (df_train["profit_lag_7"] /
                                  (df_train["Revenue"].shift(7) + 1))

df_train = df_train.dropna(subset=features)

X_tr = df_train[features]
y_rev  = df_train["Revenue"]
y_cogs = df_train["COGS"]

model_rev  = GradientBoostingRegressor(
    n_estimators=500, learning_rate=0.05,
    max_depth=4, subsample=0.8, random_state=42)
model_cogs = GradientBoostingRegressor(
    n_estimators=500, learning_rate=0.05,
    max_depth=4, subsample=0.8, random_state=42)

model_rev.fit(X_tr, y_rev)
model_cogs.fit(X_tr, y_cogs)
print("✅ Train xong")

# ── Sequential forecasting ─────────────────────────────────────
future_dates = pd.date_range(start="2023-01-01", end="2024-07-01")
results = []

for current_date in tqdm(future_dates, desc="Đang dự đoán"):
    month = current_date.month
    current_features = {
        "month":            month,
        "day_of_week":      current_date.dayofweek,
        "is_weekend":       1 if current_date.dayofweek >= 5 else 0,
        "is_special":       1 if current_date.strftime("%m-%d") in
                            ["01-01","02-14","04-30","05-01","12-31","12-24","12-25"] else 0,
        "is_payday":        1 if (current_date.day == 15 or
                            current_date + pd.offsets.MonthEnd(0) == current_date) else 0,
        "rev_lag_1":        get_lag_safe(current_date, 1,   "y",      historical_dict),
        "profit_lag_7":     get_lag_safe(current_date, 7,   "profit", historical_dict),
        "profit_lag_365":   get_lag_safe(current_date, 365, "profit", historical_dict),
        "margin_rate_lag_7": get_lag_safe(current_date, 7, "profit", historical_dict) /
                            (get_lag_safe(current_date, 7, "y", historical_dict) + 1),
    }
    for k, v in train_monthly_avg.get(month, {}).items():
        current_features[k] = v

    X_pred       = pd.DataFrame([current_features])[features]
    revenue_pred = model_rev.predict(X_pred)[0]
    cogs_pred    = model_cogs.predict(X_pred)[0]

    # Blending với same period last year
    hist_rev  = get_lag_safe(current_date, 365, "y",      historical_dict)
    hist_cogs = hist_rev - get_lag_safe(current_date, 365, "profit", historical_dict) \
                if hist_rev > 0 else 0

    final_rev  = (hist_rev  * 0.5) + (revenue_pred * 0.5)
    final_cogs = (hist_cogs * 0.5) + (cogs_pred    * 0.5)

    historical_dict[current_date] = {
        "y":      final_rev,
        "profit": final_rev - final_cogs
    }

    results.append({
        "Date":    current_date.strftime("%Y-%m-%d"),
        "Revenue": round(final_rev, 2),
        "COGS":    round(final_cogs, 2)
    })

# ── Align với sample_submission ───────────────────────────────
sample     = pd.read_csv("../dataset/sample_submission.csv", parse_dates=["Date"])
submission = pd.DataFrame(results)
submission["Date"] = pd.to_datetime(submission["Date"])

submission_final = sample[["Date"]].merge(
    submission, on="Date", how="left")

submission_final.to_csv("submission_final.csv", index=False)
print(f"✅ Done — Shape: {submission_final.shape}")
print(submission_final.head(10))

✅ Train xong


Đang dự đoán: 100%|██████████| 548/548 [00:01<00:00, 489.62it/s]

✅ Done — Shape: (548, 3)
        Date     Revenue        COGS
0 2023-01-01  2264106.77  1709313.03
1 2023-01-02  2039428.14  1522496.42
2 2023-01-03  1022728.92   773498.03
3 2023-01-04  1190927.71   896930.89
4 2023-01-05   999811.26   747220.05
5 2023-01-06  1056293.56   785180.19
6 2023-01-07  1279126.40   954193.45
7 2023-01-08  1928655.92  1471248.20
8 2023-01-09  1892506.99  1418363.96
9 2023-01-10  2286822.98  1712817.43
